In [6]:
import sqlite3
import pandas as pd

In [7]:
conn=sqlite3.connect('chicago_food_safety.db')

In [8]:
# Risk-level distribution and rate
q1="""
   WITH risk_count AS(
   SELECT Risk, COUNT(*) as total_inspections
   FROM Inspections
   GROUP BY Risk
   )
   SELECT Risk, total_inspections, ROUND(100.0*total_inspections/SUM(total_inspections) OVER (),2) AS Rate
   FROM risk_count
   Order By total_inspections DESC"""
pd.read_sql(q1,conn)

,Risk,total_inspections,Rate
0,Risk 1 (High),234013,74.48
1,Risk 2 (Medium),55779,17.75
2,Risk 3 (Low),24385,7.76


In [9]:
# Seasonal inspection patterns
q2="""
   SELECT SUBSTRING("Inspection Date",6,2) as month, COUNT(*) as total_inspections
   FROM Inspections
   GROUP BY month
   Order By month"""
pd.read_sql(q2,conn)


,month,total_inspections
0,01,24606
1,02,23158
2,03,27820
3,04,27437
4,05,28964
5,06,27853
6,07,23972
7,08,27839
8,09,27392
9,10,28249


In [10]:
# Top chains by inspection count
q3="""
   SELECT UPPER(TRIM(REPLACE("DBA Name", '''', ''))) as business_name, COUNT(*) as total_inspections
   FROM Inspections
   GROUP BY business_name
   Order By total_inspections DESC
   LIMIT 10"""
pd.read_sql(q3,conn)

,business_name,total_inspections
0,SUBWAY,4275
1,DUNKIN DONUTS,2233
2,MCDONALDS,1559
3,JIMMY JOHNS,701
4,7-ELEVEN,654
5,CHIPOTLE MEXICAN GRILL,645
6,POTBELLY SANDWICH WORKS LLC,400
7,POTBELLY SANDWICH WORKS,375
8,TACO BELL,362
9,DUNKIN DONUTS/BASKIN ROBBINS,354


In [11]:
# Monthly pass-rate trend
q4="""
   SELECT SUBSTRING("Inspection Date",6,2) as month, ROUND(100.0*(SUM(CASE WHEN Results IN ('Pass','Pass w/ Conditions') THEN 1 ELSE 0 END))/(SUM(CASE WHEN Results IN ('Pass','Pass w/ Conditions','Fail') THEN 1 ELSE 0 END)),2) AS "pass/fail %"
   FROM Inspections
   GROUP BY month
   Order By month"""
pd.read_sql(q4,conn)

,month,pass/fail %
0,01,77.68
1,02,77.56
2,03,78.06
3,04,77.66
4,05,78.20
5,06,77.22
6,07,76.69
7,08,76.03
8,09,76.63
9,10,77.25


In [12]:
# Running total of inspections per ward per year
q5="""
   WITH ward_year AS(
        SELECT l.WARD AS ward, SUBSTRING(i."Inspection Date",1,4) AS year, COUNT(*) AS inspection_count
        FROM Inspections i
        JOIN licenses l ON i."License #"=l."License Number"
        WHERE ward IS NOT NULL
        GROUP BY ward,year
    )
   SELECT ward, year,inspection_count, SUM(inspection_count) OVER(PARTITION BY ward ORDER BY year) AS running_total
   FROM ward_year
   ORDER BY ward, year"""
pd.read_sql(q5,conn)


,ward,year,inspection_count,running_total
0,1.0,2010,458,458
1,1.0,2011,461,919
2,1.0,2012,504,1423
3,1.0,2013,570,1993
4,1.0,2014,660,2653
...,...,...,...,...
845,50.0,2022,262,4399
846,50.0,2023,245,4644
847,50.0,2024,271,4915
848,50.0,2025,328,5243


In [13]:
# Top Violators within facility type
q6="""
    WITH business_violation AS (
    SELECT 
        i."DBA Name" AS business,
        i."Facility Type" AS facility_type,
        COUNT(v.Violations_code) AS violation_count
    FROM inspections i
    JOIN violations v 
        ON i."Inspection ID" = v."Inspection ID"
    WHERE i."Facility Type" IS NOT NULL
    GROUP BY facility_type, business
),
ranked AS (
    SELECT 
        facility_type,
        business,
        violation_count,
        RANK() OVER(
            PARTITION BY facility_type 
            ORDER BY violation_count DESC
        ) AS rank
    FROM business_violation
)
SELECT facility_type, business, violation_count
FROM ranked
WHERE rank = 1
ORDER BY violation_count DESC;
"""
pd.read_sql(q6,conn)

,facility_type,business,violation_count
0,Restaurant,SUBWAY,10700
1,Grocery Store,7-ELEVEN,1785
2,Bakery,ANN'S BAKERY INC,244
3,STADIUM,LEVY RESTAURANTS,238
4,GROCERY/RESTAURANT,CARNICERIA GUANAJUATO,216
...,...,...,...
429,SERVICE BAR/THEATRE,LAUGH OUT LOUD,1
430,TEMPORARY KIOSK,NATURAL BREAK,1
431,UNUSED STORAGE,CARIBBEAN AMERICAN BAKING CO,1
432,URBAN FARM,"THE ROOF CROP, LLC",1


In [14]:
# Repeat Violations Rate
q7="""WITH repeat AS (
        SELECT DISTINCT a."License #"
        FROM Inspections a
        JOIN Inspections b ON a."License #"=b."License #" AND b."Inspection Date">a."Inspection Date"
        WHERE b.Results="Fail" AND a.Results="Fail"
    ),
    any AS (
        SELECT DISTINCT "License #"
        FROM Inspections
        WHERE Results="Fail"
    )
    SELECT (SELECT COUNT(*) FROM repeat) AS repeat_count, (SELECT COUNT(*) FROM any) AS total_count,
            ROUND(100.0*(SELECT COUNT(*) FROM repeat)/(SELECT COUNT(*) FROM any),2) AS repeat_rate"""
pd.read_sql(q7,conn)

,repeat_count,total_count,repeat_rate
0,13071,25742,50.78
